In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin

train_path = "/kaggle/input/siim-isic-melanoma-classification/train.csv"
if not os.path.exists(train_path):
    train_path = "../input/siim-isic-melanoma-classification/train.csv"

test_path = "/kaggle/input/siim-isic-melanoma-classification/test.csv"
if not os.path.exists(test_path):
    test_path = "../input/siim-isic-melanoma-classification/test.csv"

sample_sub_path = (
    "/kaggle/input/siim-isic-melanoma-classification/sample_submission.csv"
)
if not os.path.exists(sample_sub_path):
    sample_sub_path = "../input/siim-isic-melanoma-classification/sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
submission_df = pd.read_csv(sample_sub_path)

expected_cols = {"image_name", "target"}
if not expected_cols.issubset(set(submission_df.columns)):
    raise ValueError(f"Sample submission must contain columns {expected_cols}")

categorical_cols = [
    "patient_id",  # now treated as regular categorical (but encoded via frequency)
    "sex",
    "anatom_site_general_challenge",
    "diagnosis",
    "benign_malignant",
]
numeric_cols = ["age_approx"]

available_features = [
    col
    for col in categorical_cols + numeric_cols
    if col in train_df.columns and col in test_df.columns
]

categorical_features = [c for c in categorical_cols if c in available_features]
numeric_features = [n for n in numeric_cols if n in available_features]

y = train_df["target"]




In [2]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        self.freq_ = X[self.column].value_counts(normalize=True).to_dict()
        self.global_freq_ = np.mean(list(self.freq_.values())) if self.freq_ else 0.0
        return self

    def transform(self, X):
        encoded = X[self.column].map(self.freq_).fillna(self.global_freq_)
        return pd.DataFrame({self.column: encoded})


# Pipeline for one‑hot encoded categorical columns (excluding patient_id)
cat_onehot_cols = [c for c in categorical_features if c != "patient_id"]
categorical_onehot_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Pipeline for patient_id using frequency encoding
if "patient_id" in categorical_features:
    patient_id_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("freqenc", FrequencyEncoder(column="patient_id")),
        ]
    )
else:
    patient_id_pipeline = None

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

transformers = []
if patient_id_pipeline is not None:
    transformers.append(("patient_id", patient_id_pipeline, ["patient_id"]))
if cat_onehot_cols:
    transformers.append(("cat_onehot", categorical_onehot_pipeline, cat_onehot_cols))
if numeric_features:
    transformers.append(("num", numeric_pipeline, numeric_features))

preprocess = ColumnTransformer(transformers=transformers)

model = LogisticRegression(
    max_iter=20000,
    solver="saga",  # suitable for high‑dimensional sparse data
    penalty="l2",
    C=50.0,  # less regularisation to capture richer features
    class_weight="balanced",
    n_jobs=5,
)

clf = Pipeline(steps=[("preprocess", preprocess), ("model", model)])

X_train, X_val, y_train, y_val = train_test_split(
    train_df[available_features],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)

val_pred = clf.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print(f"Validation AUC: {val_auc:.5f}")




IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [3]:
test_features = test_df[available_features]
test_pred = clf.predict_proba(test_features)[:, 1]

submission_df["target"] = test_pred

output_path = "submission.csv"
submission_df.to_csv(output_path, index=False, float_format="%.6f")
print(f"Submission written to {output_path}")

AttributeError: 'ColumnTransformer' object has no attribute 'transformers_'